# SHAP Comparison With And Without Imputation

This notebook reproduces the interpretation-model feature engineering from `250607_Interpretation.ipynb` and compares the top variables from SHAP analysis under two settings:

- `mean_imputation`: missing predictor values are imputed with the column mean, matching the interpretation notebook.
- `complete_case`: rows with missing predictor values are removed before model fitting.

This notebook is separate from the original interpretation workflow and does not overwrite any existing results.


In [ ]:
import pandas as pd
import numpy as np
import shap

from flaml.default import XGBClassifier as ZeroShotXGBClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer


dataset_names = ["ml_d1_predelivery", "ml_d2_earlydeath", "ml_d3_latedeath"]
top_k = 6


def prepare_interpretation_dataframe(dataset_name):
    df = pd.read_csv(f"../data/processed/{dataset_name}.csv")
    target_col = df.columns[0]

    if "_d1_" in dataset_name:
        target = [None if str(v) == "nan" else int(v == "Miscarriage or Stillbirth") for v in df[target_col].tolist()]
    else:
        target = [None if str(v) == "nan" else int(v == "Dead") for v in df[target_col].tolist()]

    df[target_col] = pd.Series(target, dtype="float")
    df = df.rename(columns={target_col: "Death"})

    if "Birthweight Measure" in df.columns:
        df = df.drop(columns=["Birthweight Measure"], errors="ignore")
    if "Type of Delivery Place" in df.columns:
        df = df.drop(columns=["Type of Delivery Place"], errors="ignore")

    bmi = np.array(df["Maternal Weight"].tolist()) / (np.array(df["Maternal Height"].tolist()) / 100) ** 2
    df["BMI"] = np.clip(bmi, 10, 50)
    df = df.drop(columns=["Maternal Weight", "Maternal Height"], errors="ignore")

    school_levels = []
    for x in df["School Level"].tolist():
        x = str(x)
        if x == "nan":
            school_levels.append(None)
        else:
            school_levels.append(int(x[0]) - 1)
    df["School Level"] = pd.Series(school_levels, dtype="float")

    if "Pre-term Delivery" in df.columns:
        preterm = []
        for x in df["Pre-term Delivery"].tolist():
            x = str(x)
            if x == "nan":
                preterm.append(None)
            else:
                preterm.append(int(x == "Preterm"))
        df["Pre-term Delivery"] = pd.Series(preterm, dtype="float")

    if "Mode of Delivery" in df.columns:
        cesarean = []
        for x in df["Mode of Delivery"].tolist():
            x = str(x)
            if x == "nan":
                cesarean.append(None)
            else:
                cesarean.append(int(x == "Cesarean"))
        df["Mode of Delivery Cesarea"] = pd.Series(cesarean, dtype="float")
        df = df.drop(columns=["Mode of Delivery"], errors="ignore")

    if "Baby Sex" in df.columns:
        baby_sex_female = []
        for x in df["Baby Sex"].tolist():
            x = str(x)
            if x == "nan":
                baby_sex_female.append(None)
            else:
                baby_sex_female.append(int(x == "Female"))
        df["Baby Sex Female"] = pd.Series(baby_sex_female, dtype="float")
        df = df.drop(columns=["Baby Sex"], errors="ignore")

    if "Delivery Place" in df.columns:
        delivery_home = []
        for x in df["Delivery Place"].tolist():
            x = str(x)
            if x == "nan":
                delivery_home.append(None)
            else:
                delivery_home.append(int(x in ["Home", "Other"]))
        df["Delivery Place Home"] = pd.Series(delivery_home, dtype="float")
        df = df.drop(columns=["Delivery Place"], errors="ignore")

    df = df[df["Death"].notnull()].copy()
    df["Death"] = df["Death"].astype(int)
    return df


def build_model(model_class):
    model = model_class()
    params = model.get_params()
    updates = {}

    if "random_state" in params:
        updates["random_state"] = 42
    if "seed" in params:
        updates["seed"] = 42
    if "n_jobs" in params:
        updates["n_jobs"] = -1
    if "eval_metric" in params:
        updates["eval_metric"] = "auc"
    if "use_label_encoder" in params:
        updates["use_label_encoder"] = False

    if updates:
        model.set_params(**updates)
    return model


def prepare_features(df, mode):
    X = df.iloc[:, 1:].copy()
    y = df.iloc[:, 0].copy()

    if mode == "mean_imputation":
        imputer = SimpleImputer(strategy="mean")
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    elif mode == "complete_case":
        valid_mask = X.notna().all(axis=1)
        X = X.loc[valid_mask].copy()
        y = y.loc[valid_mask].copy()
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return X, y.astype(int)


def get_shap_importance(model_class, df, mode):
    X, y = prepare_features(df, mode)
    model = build_model(model_class)
    model.fit(X, y)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X)
    importance = pd.DataFrame({
        "feature": X.columns,
        "mean_abs_shap": np.abs(shap_values.values).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False, ignore_index=True)
    return importance, len(X)


def compare_top_features(imputed_importance, complete_case_importance, dataset_name, top_k=10):
    top_imputed = imputed_importance.head(top_k).copy()
    top_complete = complete_case_importance.head(top_k).copy()
    imputed_features = list(top_imputed["feature"])
    complete_features = list(top_complete["feature"])
    shared_features = [feature for feature in imputed_features if feature in complete_features]

    comparison = pd.DataFrame({
        "dataset_name": dataset_name,
        "top_k": top_k,
        "same_top_k_set": set(imputed_features) == set(complete_features),
        "n_shared_top_k": len(shared_features),
        "shared_top_k_features": [", ".join(shared_features)],
        "imputed_top_k": [", ".join(imputed_features)],
        "complete_case_top_k": [", ".join(complete_features)],
    })
    return comparison


In [11]:
all_importance_tables = []
all_comparisons = []

for dataset_name in dataset_names:
    model_class = XGBClassifier if "_d1_" in dataset_name else ZeroShotXGBClassifier
    df = prepare_interpretation_dataframe(dataset_name)

    imputed_importance, imputed_n = get_shap_importance(model_class, df, mode="mean_imputation")
    imputed_importance.insert(0, "mode", "mean_imputation")
    imputed_importance.insert(0, "n_rows_used", imputed_n)
    imputed_importance.insert(0, "dataset_name", dataset_name)
    all_importance_tables.append(imputed_importance)

    complete_importance, complete_n = get_shap_importance(model_class, df, mode="complete_case")
    complete_importance.insert(0, "mode", "complete_case")
    complete_importance.insert(0, "n_rows_used", complete_n)
    complete_importance.insert(0, "dataset_name", dataset_name)
    all_importance_tables.append(complete_importance)

    comparison = compare_top_features(
        imputed_importance[["feature", "mean_abs_shap"]],
        complete_importance[["feature", "mean_abs_shap"]],
        dataset_name=dataset_name,
        top_k=top_k,
    )
    comparison.insert(1, "n_rows_mean_imputation", imputed_n)
    comparison.insert(2, "n_rows_complete_case", complete_n)
    all_comparisons.append(comparison)

importance_df = pd.concat(all_importance_tables, ignore_index=True)
comparison_df = pd.concat(all_comparisons, ignore_index=True)

display(comparison_df)
display(importance_df)


,dataset_name,n_rows_mean_imputation,n_rows_complete_case,top_k,same_top_k_set,n_shared_top_k,shared_top_k_features,imputed_top_k,complete_case_top_k
0,ml_d1_predelivery,10596,8358,11,True,11,"Gestation, BMI, Maternal Age, Years of Educati...","Gestation, BMI, Maternal Age, Years of Educati...","Gestation, BMI, Maternal Age, Years of Educati..."
1,ml_d2_earlydeath,10409,8062,11,True,11,"Birthweight, Cord care Chlorhexidine, Gestatio...","Birthweight, Cord care Chlorhexidine, Gestatio...","Birthweight, Bag and Mask Resuscitation, Gesta..."
2,ml_d3_latedeath,10288,7974,11,True,11,"Birthweight, Gestation, BMI, Maternal Age, Yea...","Birthweight, Gestation, BMI, Maternal Age, Yea...","Birthweight, Gestation, BMI, Maternal Age, Par..."


,dataset_name,n_rows_used,mode,feature,mean_abs_shap
0,ml_d1_predelivery,10596,mean_imputation,Gestation,0.810163
1,ml_d1_predelivery,10596,mean_imputation,BMI,0.688555
2,ml_d1_predelivery,10596,mean_imputation,Maternal Age,0.457806
3,ml_d1_predelivery,10596,mean_imputation,Years of Education,0.317127
4,ml_d1_predelivery,10596,mean_imputation,Antenatal Visits,0.267200
...,...,...,...,...,...
93,ml_d3_latedeath,7974,complete_case,Mode of Delivery Cesarea,0.007779
94,ml_d3_latedeath,7974,complete_case,Oxygen,0.003143
95,ml_d3_latedeath,7974,complete_case,CPAP,0.002273
96,ml_d3_latedeath,7974,complete_case,Dexamethasone,0.002026


In [8]:
x = importance_df[importance_df["dataset_name"] == "ml_d2_earlydeath"]

In [9]:
x

,dataset_name,n_rows_used,mode,feature,mean_abs_shap
22,ml_d2_earlydeath,10409,mean_imputation,Birthweight,0.423573
23,ml_d2_earlydeath,10409,mean_imputation,Cord care Chlorhexidine,0.400419
24,ml_d2_earlydeath,10409,mean_imputation,Gestation,0.365053
25,ml_d2_earlydeath,10409,mean_imputation,Bag and Mask Resuscitation,0.316384
26,ml_d2_earlydeath,10409,mean_imputation,Maternal Age,0.294123
27,ml_d2_earlydeath,10409,mean_imputation,BMI,0.291317
28,ml_d2_earlydeath,10409,mean_imputation,Oxygen,0.259714
29,ml_d2_earlydeath,10409,mean_imputation,Years of Education,0.145479
30,ml_d2_earlydeath,10409,mean_imputation,Parity,0.138339
31,ml_d2_earlydeath,10409,mean_imputation,Antenatal Visits,0.133224
